# Dyson Protocol – Make Dwapps, Get Paid

**Host Python scripts, serve decentralized websites, and run scheduled tasks with trustless, censorship-resistant execution. Trade names in a dynamic on-chain market, mint custom tokens and NFTs, and store arbitrary data—all fully on-chain.**

---

## What & Why

- **Problem**  
  - Blockchain DApp UIs still load from centralized servers—developers host them off-chain, and end-users can't self-host or audit the code.

- **Solution**  
  - Store HTML/CSS/JS assets in the chain's storage so browsers load UI from the ledger.  
  - Push application logic on-chain and execute periodic jobs (crontasks) without any off-chain trigger.  
  - Run a dynamic on-chain name market using Harberger-style fees.  
  - Mint custom tokens and NFT classes based on on-chain names.  
  - Store arbitrary data in the chain’s storage module.

- **Key Use Cases**  
  - **Autonomous payouts**: schedule hourly dividend distributions without users having to claim.  
  - **Timed auctions**: start and end bids exactly on-chain, with no external cron.  
  - **Game rounds**: progress players automatically through time-boxed stages.  
  - **Price oracles**: post market data at fixed intervals, fully on-chain.  
  - **Nameservice-driven assets**: register and trade domain-backed NFTs in a live marketplace.

- **Outcome**  
  - **A spectrum of security**: from fully trustless script and web UI, to fully centralized, depending on your needs.


## Installation




### 0. Build the dysvm dependencies
Only do this once.

In [ ]:
%%bash
make dysvm 

### 1. Build the Dyson Protocol binary


In [ ]:
%%bash
make install

Installing dysond binary...
build_tags: netgo,app_v1
commit: 132ac87
cosmos_sdk_version: v0.53.0
go: go version go1.24.3 darwin/arm64
name: dyson
server_name: dysond
version: develop



In [8]:
%%bash
dysond version --long | tail


- rsc.io/qr@v0.2.0
- sigs.k8s.io/yaml@v1.6.0
build_tags: netgo,app_v1
commit: 132ac87
cosmos_sdk_version: v0.53.0
go: go version go1.24.3 darwin/arm64
name: dyson
server_name: dysond
version: develop



### 2. Create new accounts

In [10]:
%%bash
dysond keys add alice 

- address: dys21ldyd2ngz4ttkmvud40pshksz9g0yx9lzjy9py5
  name: alice
  pubkey: '{"@type":"/cosmos.crypto.secp256k1.PubKey","key":"AjenBZPnfrfFQtvFT1KiGI5YFfKAENEfPLOZafBlPwuN"}'
  type: local




**Important** write this mnemonic phrase in a safe place.
It is the only way to recover your account if you ever forget your password.

useful garbage divorce found surface like jump oven bitter maze ranch switch stomach rough head soap front infant camera twin renew casino olive spot


### 2. Update the On-chain Python Script

This example uploads a Python script that demonstrates storage operations. The full script is available at [examples/storage_example.py](examples/storage_example.py).

**Key Functions in the Script** (excerpt):

```python
def save_message(message):
    # the account that signed the transaction
    caller = get_executor_address()
    _msg({"@type":"/dysonprotocol.storage.v1.MsgStorageSet","owner": get_script_address() ,"index":f"greetings/{caller}","data": json.dumps({"greeting": message})})

def wsgi(environ, start_response):
    # Define response status and headers
    status_code = "200 OK"
    headers = [("Content-Type", "text/html")]
    start_response(status_code, headers)

    # Prepare the query parameters
    query_params = {
        "@type":"/dysonprotocol.storage.v1.QueryStorageListRequest",
        "owner": get_script_address(),
        "index_prefix":"greetings/"
    }
    
# ... more code in the full example ...
```

Let's see the full script:

In [11]:
%%bash
cat examples/storage_example.py

import json
from html import escape
from dys import get_script_address, get_executor_address, _msg, _query


def save_message(message):
    # the account that signed the transaction
    caller = get_executor_address()
    return _msg(
        {
            "@type": "/dysonprotocol.storage.v1.MsgStorageSet",
            "owner": get_script_address(),
            "index": f"greetings/{caller}",
            "data": json.dumps({"greeting": message}),
        }
    )


def wsgi(environ, start_response):
    # Define response status and headers
    status_code = "200 OK"
    headers = [("Content-Type", "text/html")]
    start_response(status_code, headers)

    # Prepare the query parameters
    query_params = {
        "@type": "/dysonprotocol.storage.v1.QueryStorageListRequest",
        "owner": get_script_address(),
        "index_prefix": "greetings/",
    }

    # Get messages from storage
    storage_result = _query(query_params)

    # Start building HTML output
    output = "<html><b

Now let's upload the script to the chain using Alice's address:

In [16]:
%%bash
ALICE_ADDRESS=$(dysond keys show -a alice)
dysond tx script update --from alice -y -o json --gas 500000 --code "$(cat examples/storage_example.py)" |  dysond q wait-tx -o json | jq '{height, txhash, code, gas_wanted, gas_used, "script_version": .events[] | select(.type=="dysonprotocol.script.v1.EventUpdateScript") | .attributes[] | select(.key=="version") | .value}'


Usage:
  dysond tx script update [--code <code> | --code-path <path to source code>] [flags]

Flags:
  -a, --account-number uint         The account number of the signing account (offline mode only)
      --aux                         Generate aux signer data instead of sending a tx
  -b, --broadcast-mode string       Transaction broadcasting mode (sync|async) (default "sync")
      --chain-id string             The network chain ID
      --code string                 Source code as a string
      --code-path string            Path to the source code file
      --dry-run                     ignore the --gas flag and perform a simulation of a transaction, but don't broadcast it (when enabled, the local Keybase is not accessible)
      --fee-granter string          Fee granter grants fees for the transaction
      --fee-payer string            Fee payer pays fees for the transaction instead of deducting from the signer
      --fees string                 Fees to pay along with transactio

### 3. Execute the Script Function

Invoke the `save_message` function using Bob's account, passing `"my name is bob"` as an argument:

In [13]:
%%bash
ALICE_ADDRESS=$(dysond keys show -a alice)
# Save the message to the storage
dysond tx script exec-script --from bob --script-address $ALICE_ADDRESS --function-name save_message --args '["my name is <b>bob</b>"]' -y  | dysond query wait-tx -o json | ./scripts/parse_exec_script_tx.py | jq 

Usage:
  dysond query wait-tx [hash] [flags]

Aliases:
  wait-tx, event-query-tx-for

Examples:
By providing the transaction hash:
$ dysond q wait-tx [hash]

Or, by piping a "tx" command:
$ dysond tx [flags] | dysond q wait-tx


Flags:
      --grpc-addr string   the gRPC endpoint to use for this chain
      --grpc-insecure      allow gRPC over insecure channels, if not the server must use TLS
      --height int         Use a specific height to query state at (this can error if the node is pruning state)
  -h, --help               help for wait-tx
      --node string        <host>:<port> to CometBFT RPC interface for this chain (default "tcp://localhost:26657")
  -o, --output string      Output format (text|json) (default "text")
      --timeout duration   The maximum time to wait for the transaction to be included in a block (default 15s)

Global Flags:
      --home string         directory for config and data (default "/Users/user/.dysonprotocol")
      --log_format string   The loggi

CalledProcessError: Command 'b'ALICE_ADDRESS=$(dysond keys show -a alice)\n# Save the message to the storage\ndysond tx script exec-script --from bob --script-address $ALICE_ADDRESS --function-name save_message --args \'["my name is <b>bob</b>"]\' -y  | dysond query wait-tx -o json | ./scripts/parse_exec_script_tx.py | jq \n'' returned non-zero exit status 5.

### 4. Query the WSGI Endpoint

Finally, confirm the data is stored and accessible via an HTTP request to the script's WSGI endpoint:

In [ ]:
%%bash
ALICE_ADDRESS=$(dysond keys show -a alice)
DWAPP_SERVER_ADDRESS=$(dysond config get app dwapp.address | tr -d '"')
DWAPP_URL="http://$ALICE_ADDRESS.$DWAPP_SERVER_ADDRESS/some-path?query=some-query"
curl -v $DWAPP_URL

## Notes & Edge Cases
- Always escape user generated content when rendering it in the browser.
- Ensure that you have a valid account (e.g., `alice`, `bob`) with sufficient balance to pay for gas fees.
- Always verify that you're interacting with the right script address.
- Make sure to provide sufficient gas for script updates (as seen in the example, we used `--gas 500000`).

## Conclusion

This example demonstrates how to:
1. Update on-chain Python code.
2. Execute a function that stores data on the Dyson Protocol.
3. Retrieve data via a WSGI endpoint.

Feel free to adapt the `save_message` function or the WSGI application for more advanced use cases, such as multi-key storage or complex business logic.

## More Documentation

For more detailed information about specific modules, please refer to the following documentation:

### Module Guides

- [Script Module](notebooks/scripting_guide.md): Comprehensive guide to the Script module for on-chain Python execution
- [Storage Module](notebooks/storage_guide.md): Detailed documentation on the Storage module for on-chain data persistence
- [Crontask Module](notebooks/crontask_guide.md): Complete guide to the Crontask module for scheduled transaction execution
- [Nameservice Module](notebooks/nameservice_guide.md): Guide to the Nameservice module for registering names and creating NFTs
- [DysLang Guide](notebooks/dyslang_guide.md): Complete programming reference for the Dyson Language

### Interactive Notebooks

The same guides are also available as interactive Jupyter notebooks in the `notebooks/` directory:

- [Script Module Notebook](notebooks/scripting_guide.ipynb)
- [Storage Module Notebook](notebooks/storage_guide.ipynb)
- [Crontask Module Notebook](notebooks/crontask_guide.ipynb)
- [Nameservice Module Notebook](notebooks/nameservice_guide.ipynb)
- [DysLang Guide Notebook](notebooks/dyslang_guide.ipynb)

### Code Examples

Explore practical examples in the `examples/` directory:

- [Storage Example](examples/storage_example.py): Basic storage operations and WSGI endpoint
- [Crontask Example](examples/crontask_countdown.py): Scheduled task countdown implementation
- [Crontask Script](examples/crontask_script.py): Advanced scheduled transaction execution
- [DysLang Example](examples/dyslang_example.py): Comprehensive language feature demonstration
- [Balance Example](examples/balance_example.py): Account balance querying
- [WSGI Example](examples/simple_wsgi_example.py): Simple web application server
- [AST Explorer](examples/ast_explorer.py): Python Abstract Syntax Tree exploration
- [ICA Example](examples/ica_e2e.py): Inter-Chain Account end-to-end example
- [ICA Module](examples/ica.py): Inter-Chain Account implementation
- [Script Query Height](examples/script_query_height.py): Query blockchain height from scripts
